# EV Purchase Intention Analysis (English Results Report)

[中文 Notebook](01_ev_purchase_intention_zh.ipynb) · [English Notebook](01_ev_purchase_intention_en.ipynb) · [Project README](../README.en.md)

This notebook is a **results and provenance report**, not a second analysis implementation. It uses the shared question mapping and reads one completed run directory. It does not refit models, rerun bootstrap, or retune the forest in the presentation layer.

## Research question

Q21 is the ordinal willingness-to-pay outcome. T (Q15/Q16) and V (Q22–Q25) are fixed project-defined proxies. Ordered logit is the primary analysis; five bootstrap paths, nested heterogeneity LR tests, out-of-fold machine learning, and SHAP are supplementary. Results are cross-sectional conditional associations or predictive explanations, not causal estimates.


## 1. Environment and fixed run directory

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Locate the repository even when Jupyter starts in notebooks/.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / "main.py").exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_DATA_PATH, MODEL_SPECS, QUESTION_TEXT
from src.data_schema import audit_data, build_analysis_frame, load_raw_data, mapping_manifest

DATA_PATH = Path(DEFAULT_DATA_PATH)
# Set RUN_ID to a validated directory when publishing a fixed report.
# None selects one complete local run and keeps every artifact in that same directory.
RUN_ID = None
_required_artifacts = {
    "run_metadata.json", "audit.json", "ordered_logit_coefficients.csv",
    "mediation_paths.csv", "heterogeneity_results.csv", "ml_summary.csv",
    "ml_run_metadata.json", "shap_importance.csv",
}
_run_candidates = [
    p for p in sorted((PROJECT_ROOT / "figures" / "runs").glob("run-*"))
    if _required_artifacts.issubset({item.name for item in p.iterdir()})
]
if RUN_ID is None:
    if not _run_candidates:
        raise FileNotFoundError(
            "No complete saved run found. First run `.venv/bin/python main.py --analysis all`, "
            "then reopen this notebook."
        )
    RUN_DIR = _run_candidates[-1]
else:
    RUN_DIR = PROJECT_ROOT / "figures" / "runs" / RUN_ID
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Run directory does not exist: {RUN_DIR}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Data file:    {DATA_PATH}")
print(f"Run selected: {RUN_DIR.name}")


## 2. Fixed question map and specifications

In [ ]:
raw = load_raw_data(DATA_PATH)
primary = build_analysis_frame(raw, specification="primary")
manifest = mapping_manifest()

spec_table = pd.DataFrame([
    {
        "specification": name,
        "technology_questions": ", ".join(f"Q{q}" for q in spec["technology_questions"]),
        "value_questions": ", ".join(f"Q{q}" for q in spec["value_questions"]),
        "controls": spec["controls_encoding"],
        "description": spec["description"],
    }
    for name, spec in MODEL_SPECS.items()
])
display(spec_table)

display(pd.DataFrame({
    "derived_variable": ["Y", "T", "V", "M1", "M2"],
    "question_items": ["Q21", "Q15 + Q16", "Q22–Q25", "Q18", "Q19"],
    "role": ["ordinal outcome", "core proxy", "core proxy", "exploratory mediator", "exploratory mediator"],
}))
print(f"Raw shape: {raw.shape}; primary analysis frame shape: {primary.shape}")


## 3. Data audit and provenance

In [ ]:
metadata = json.loads((RUN_DIR / "run_metadata.json").read_text(encoding="utf-8"))
audit = json.loads((RUN_DIR / "audit.json").read_text(encoding="utf-8"))

provenance = pd.DataFrame([{
    "run": RUN_DIR.name,
    "n_rows": audit.get("n_rows"),
    "n_columns": audit.get("n_columns"),
    "duplicate_rows": audit.get("duplicate_rows"),
    "historical_processing_columns": audit.get("historical_processing_columns"),
    "data_sha256": metadata.get("data_sha256"),
    "random_seed": metadata.get("random_seed"),
    "git_commit": metadata.get("git", {}).get("commit"),
    "git_dirty": metadata.get("git", {}).get("dirty"),
}])
display(provenance.T.rename(columns={0: "value"}))

composite_rows = []
for name, item in audit.get("composites", {}).items():
    composite_rows.append({
        "composite": name,
        "questions": ", ".join(f"Q{q}" for q in item.get("questions", [])),
        "complete_rows": item.get("complete_rows"),
        "missing_rows": item.get("missing_rows"),
        "mean": item.get("mean"),
        "cronbach_alpha": item.get("cronbach_alpha"),
    })
display(pd.DataFrame(composite_rows))


## 4. Ordered-logit associations

In [ ]:
coef = pd.read_csv(RUN_DIR / "ordered_logit_coefficients.csv")
diag = json.loads((RUN_DIR / "ordered_logit_diagnostics.json").read_text(encoding="utf-8"))

predictors = coef[(coef["term_type"] == "predictor") & coef["term"].isin(["T", "V"])].copy()
predictors["odds_ratio_ci_low"] = np.exp(predictors["ci_low"])
predictors["odds_ratio_ci_high"] = np.exp(predictors["ci_high"])
show = predictors[["specification", "model", "term", "coefficient", "odds_ratio", "odds_ratio_ci_low", "odds_ratio_ci_high", "p_value", "n_obs"]].copy()
display(show.style.format({
    "coefficient": "{:.3f}", "odds_ratio": "{:.3f}",
    "odds_ratio_ci_low": "{:.3f}", "odds_ratio_ci_high": "{:.3f}",
    "p_value": "{:.3g}",
}))

convergence = []
for spec, models in diag.items():
    for model_name, item in models.items():
        convergence.append({
            "specification": spec,
            "model": model_name,
            "converged": item.get("converged"),
            "n_obs": item.get("n_obs"),
            "aic": item.get("aic"),
            "bic": item.get("bic"),
        })
display(pd.DataFrame(convergence))


## 5. Exploratory bootstrap paths

In [ ]:
mediation = pd.read_csv(RUN_DIR / "mediation_paths.csv")
mediation_view = mediation[[
    "x", "mediator", "n_obs", "indirect", "ci_low", "ci_high",
    "bootstrap_iterations", "bootstrap_valid", "bootstrap_failures"
]].copy()
display(mediation_view.style.format({
    "indirect": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}"
}))

# The saved plot is generated by the analysis module; this cell only displays it.
plot_path = RUN_DIR / "mediation_bootstrap_distributions.png"
if plot_path.exists():
    display(Image(filename=str(plot_path)))


## 6. Heterogeneity results

In [ ]:
hetero = pd.read_csv(RUN_DIR / "heterogeneity_results.csv")
display(hetero[[
    "group_var", "n_obs", "n_groups", "df_diff", "lr_stat",
    "p_raw", "p_holm", "status"
]].style.format({"lr_stat": "{:.3f}", "p_raw": "{:.3g}", "p_holm": "{:.3g}"}))
print("Use p_holm for the five-comparison interpretation; group codes remain questionnaire codes.")


## 7. Machine learning and SHAP

In [ ]:
ml_summary = pd.read_csv(RUN_DIR / "ml_summary.csv")
summary_view = ml_summary[[
    "feature_set_name", "model", "accuracy_mean", "macro_f1_mean",
    "qwk_mean", "ordinal_mae_mean"
]].copy()
display(summary_view.style.format({
    "accuracy_mean": "{:.3f}", "macro_f1_mean": "{:.3f}",
    "qwk_mean": "{:.3f}", "ordinal_mae_mean": "{:.3f}"
}))

# Compact, reproducible comparison figure from the saved fold summary.
plot_df = ml_summary[ml_summary["model"].isin(["majority", "ordered_logit", "random_forest"])].copy()
fig, ax = plt.subplots(figsize=(9, 4.5))
for model_name, group in plot_df.groupby("model"):
    ax.plot(group["feature_set_name"], group["qwk_mean"], marker="o", label=model_name)
ax.axhline(0, color="#777", linewidth=0.8)
ax.set_ylabel("Quadratic weighted kappa (mean)")
ax.set_xlabel("Feature set")
ax.set_title("Out-of-fold ordinal prediction")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

ml_meta = json.loads((RUN_DIR / "ml_run_metadata.json").read_text(encoding="utf-8"))
print("SHAP status:", ml_meta.get("shap", {}).get("status"))
print("SHAP target:", ml_meta.get("shap", {}).get("target"))
shap_path = RUN_DIR / "shap_importance.png"
if shap_path.exists():
    display(Image(filename=str(shap_path)))


## 8. Artifact integrity checks

In [ ]:
expected_files = [
    "run_metadata.json", "audit.json", "ordered_logit_coefficients.csv",
    "ordered_logit_diagnostics.json", "mediation_paths.csv",
    "heterogeneity_results.csv", "ml_summary.csv", "ml_oof_predictions.csv",
    "ml_run_metadata.json", "shap_importance.csv", "shap_importance.png",
]
checks = pd.DataFrame({
    "artifact": expected_files,
    "exists": [(RUN_DIR / name).exists() for name in expected_files],
})
display(checks)

# Verify that the selected run refers to the same local data file.
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

recorded_sha = metadata.get("data_sha256")
current_sha = sha256(DATA_PATH)
print("Data SHA recorded:", recorded_sha)
print("Data SHA current:  ", current_sha)
print("Data hash matches:  ", recorded_sha == current_sha)
failed_folds = pd.read_csv(RUN_DIR / "ml_fold_metrics.csv")
print("Failed ML folds:", int((failed_folds["status"] != "ok").sum()))


## 9. Interpretation limits

- Odds ratios are conditional associations after adjustment, not causal effects.
- A bootstrap interval excluding zero supports an exploratory indirect association in this sample; it does not justify full/partial mediation labels.
- Holm-adjusted p-values are the relevant heterogeneity comparison; an isolated raw p-value is not stable subgroup evidence.
- QWK, accuracy, and SHAP describe held-out prediction, not causal validation of the economic model.
- When changing `RUN_ID`, recheck the data SHA, Git state, and that every artifact comes from the same run directory.
